# CSV Basics — 08: CSV Data Transformations

## What you will learn
Raw CSV data almost always needs cleaning before it's useful.
This notebook covers the most common transformation patterns.

In this notebook:
1. Trimming whitespace from values and column names
2. Normalizing column names (lowercase, no spaces)
3. Type casting and validation
4. Splitting and merging columns
5. Regex extraction from messy text fields
6. Complete data cleaning pipeline


In [ ]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

In [ ]:
import pathlib

# Realistic messy CSV from a legacy system
messy_csv = '''"Order ID ","  Customer Name  "," Product  "," Amount $","  Qty ","  Date  "," Status "
" 1 "," ALICE SMITH  "," Laptop Pro 15\"  "," $1,299.99 "," 2 "," Jan 15, 2024 "," Confirmed "
" 2 "," Bob O'Brien  "," USB-C Hub (3-port) "," $49.99 "," 5 "," Feb 2, 2024 "," SHIPPED "
" 3 "," carol garcia  "," Monitor 27\" 4K "," $799.00 "," 1 "," March 1 2024 "," pending "
" 4 "," Dave  Lee  "," Keyboard+Mouse Kit "," $129.99 "," 3 "," 2024-04-15 "," DELIVERED "
'''

messy_path = f"{DATA_DIR}/messy.csv"
pathlib.Path(messy_path).write_text(messy_csv)
print("Created messy CSV with common real-world issues:")
print("  - Column names have spaces and special chars")
print("  - String values wrapped in extra quotes and spaces")
print("  - Amount has $ sign and comma as thousands separator")
print("  - Date in multiple formats")
print("  - Status has mixed case")
print("  - Customer names have extra spaces")

## Step 1 — Clean Column Names


In [ ]:
import re

# Read raw
df_raw = spark.read.csv(messy_path, header=True,
                         ignoreLeadingWhiteSpace=True,
                         ignoreTrailingWhiteSpace=True)
print("Raw column names:")
for col in df_raw.columns:
    print(f"  '{col}'")

# Normalize column names: lowercase, strip, replace spaces with _
def normalize_col_name(name):
    name = name.strip().lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)  # replace non-alphanumeric with _
    name = name.strip('_')                    # remove leading/trailing _
    return name

clean_names = [normalize_col_name(c) for c in df_raw.columns]
df = df_raw.toDF(*clean_names)

print("\nNormalized column names:")
for col in df.columns:
    print(f"  '{col}'")
df.show(truncate=False)

## Step 2 — Clean String Values: Trim, Case, Strip Special Chars


In [ ]:
# Trim customer names and normalize case
df = df.withColumn("customer_name",
    F.initcap(                                          # Title Case
        F.regexp_replace(
            F.trim(F.col("customer_name")),             # strip spaces
            r"\s+", " "                               # collapse multiple spaces
        )
    )
).withColumn("status",
    F.lower(F.trim(F.col("status")))                   # lowercase + trim
)

print("After name and status normalization:")
df.select("customer_name","status").show()

In [ ]:
# Clean amount: remove $, commas → cast to double
df = df.withColumn("amount",
    F.regexp_replace(F.trim(F.col("amount__")), r"[$,\s]", "").cast("double")
)

# Clean qty
df = df.withColumn("qty",
    F.trim(F.col("qty")).cast("integer")
)

print("After amount and qty cleaning:")
df.select("order_id","amount__","amount","qty").show()

## Step 3 — Parse Multiple Date Formats


In [ ]:
# We have 4 date formats in one column: "Jan 15, 2024", "Feb 2, 2024", "March 1 2024", "2024-04-15"
from pyspark.sql.functions import coalesce, to_date

def parse_flexible_date(col_name):
    """Try multiple date formats, return first match."""
    formats = [
        "yyyy-MM-dd",           # 2024-04-15
        "MMM dd, yyyy",         # Jan 15, 2024
        "MMM d, yyyy",          # Feb 2, 2024
        "MMMM d yyyy",          # March 1 2024
        "MMM dd yyyy",          # Jan 15 2024
        "MM/dd/yyyy",           # 01/15/2024
        "dd/MM/yyyy",           # 15/01/2024
    ]
    return coalesce(*[to_date(F.trim(col_name), fmt) for fmt in formats])

df = df.withColumn("order_date", parse_flexible_date(F.col("date")))

print("Date parsing (multiple formats → single DATE column):")
df.select("date","order_date").show(truncate=False)

## Step 4 — Split Compound Columns


In [ ]:
# Split "product" column: product name + size spec
# Examples: "Laptop Pro 15"", "Monitor 27" 4K", "USB-C Hub (3-port)"
df = df.withColumn("product_name",
    F.regexp_replace(F.trim(F.col("product")), r'[\"]', "")
).withColumn("product_size",
    F.regexp_extract(F.col("product"), r'(\d+[\"]|\d+K|\.\d+)', 1)
)

print("Split product into name + size:")
df.select("product","product_name","product_size").show(truncate=False)

## Step 5 — Complete Cleaned DataFrame


In [ ]:
# Final clean DataFrame
df_clean = df.select(
    F.trim(F.col("order_id")).cast("integer").alias("order_id"),
    F.col("customer_name"),
    F.col("product_name"),
    F.col("product_size"),
    F.col("amount"),
    F.col("qty"),
    F.col("order_date"),
    F.col("status"),
    # Derived columns
    (F.col("amount") * F.col("qty")).alias("line_total"),
    F.current_timestamp().alias("_processed_at"),
).filter(F.col("order_id").isNotNull())

print("Final cleaned DataFrame:")
df_clean.printSchema()
df_clean.show(truncate=False)

# Save as Parquet
df_clean.write.mode("overwrite").parquet(f"{DATA_DIR}/cleaned_orders")
print(f"\nSaved {df_clean.count()} clean rows to Parquet")

## Summary: CSV Transformation Patterns

```python
# Normalize column names
clean_names = [re.sub(r'[^a-z0-9]+','_', c.strip().lower()).strip('_')
               for c in df.columns]
df = df.toDF(*clean_names)

# Trim + normalize case
F.initcap(F.regexp_replace(F.trim(col), r"\s+", " "))

# Clean amount: remove $, comma → double
F.regexp_replace(F.trim(col), r"[$,\s]", "").cast("double")

# Multi-format date parsing
coalesce(*[to_date(col, fmt) for fmt in ["yyyy-MM-dd", "MMM dd, yyyy", ...]])

# Split on delimiter
F.split(col, "-").getItem(0)   # first part
F.split(col, "-").getItem(1)   # second part

# Regex extract
F.regexp_extract(col, r"(\d+)", 1)   # first number

# Regex replace
F.regexp_replace(col, r"[^a-zA-Z0-9]", "")  # keep only alphanumeric
```
